# Crime Text NLP Pipeline

In [1]:
import pandas as pd
import numpy as np
from textblob import TextBlob
import re


/Users/nadiacamacho/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


ModuleNotFoundError: No module named 'textblob'

In [ ]:
# Reading in the text file.
# The file contains one JSON record per line.

text_df = pd.read_json(
    '/Users/nadiacamacho/Downloads/CrimeReport (1)(1).txt',
    lines=True
)

# Looking at what was loaded into the DataFrame.
text_df.head()

In [ ]:
# Looking at the columns stored in the DataFrame.
text_df.columns

In [ ]:
# Pulling out the main fields we need.

clean_df = pd.DataFrame()
clean_df['Text_ID'] = ['TXT_' + str(i+1) for i in range(len(text_df))]
clean_df['Raw_Text'] = text_df['text']
clean_df['Date'] = text_df['created_at']

# Extracting the username from the nested user column.
clean_df['User'] = text_df['user'].apply(
    lambda x: x.get('screen_name', 'Unknown') if isinstance(x, dict) else 'Unknown'
)

# Extracting the location from the nested place column.
clean_df['Location'] = text_df['place'].apply(
    lambda x: x.get('full_name', 'Unknown') if isinstance(x, dict) else 'Unknown'
)

clean_df.head()

In [ ]:
# Cleaning the text by removing links and extra spaces.

def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

clean_df['Clean_Text'] = clean_df['Raw_Text'].apply(clean_text)
clean_df[['Raw_Text','Clean_Text']].head()

In [ ]:
# Finding the sentiment of each post.

def get_sentiment(text):
    polarity = TextBlob(text).sentiment.polarity

    if polarity > 0:
        return 'Positive'
    elif polarity < 0:
        return 'Negative'
    else:
        return 'Neutral'

clean_df['Sentiment'] = clean_df['Clean_Text'].apply(get_sentiment)
clean_df['Sentiment'].value_counts()

In [ ]:
# Assigning a topic to each post using keywords.

def assign_topic(text):
    text = text.lower()

    if any(word in text for word in ['shoot', 'assault', 'violence', 'murder']):
        return 'Assault / Violence'
    elif any(word in text for word in ['robbery', 'theft', 'heist', 'steal']):
        return 'Theft / Robbery'
    elif any(word in text for word in ['fire', 'arson']):
        return 'Fire / Arson'
    elif any(word in text for word in ['accident', 'traffic', 'crash']):
        return 'Traffic Accident'
    elif any(word in text for word in ['police', 'crime', 'arrest']):
        return 'Public Disturbance'
    else:
        return 'Other'

clean_df['Topic'] = clean_df['Clean_Text'].apply(assign_topic)
clean_df['Topic'].value_counts()

In [ ]:
# Creating the final structured output.

final_df = clean_df[['Text_ID','Clean_Text','Topic','Sentiment','Location']]
final_df.head()

In [ ]:
# Saving the final output as a CSV file.

final_df.to_csv('Text.csv', index=False)

print('CSV file saved successfully.')